# Booked ≠ Stayed: Where Hotel Booking Reliability Breaks Down

**Consultancy brief.** A hotel group wants to reduce unreliable demand without imposing blanket restrictions on reliable customers. This report identifies where booking failure is concentrated and separates **likelihood of cancellation** from **failure close to arrival**.

**Audience:** Revenue Director and Commercial Director  
**Dataset:** Hotel Booking Demand, 119,390 records, July 2015–August 2017  
**URL:** https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

## Exploratory analysis and reproducibility trail

The final explanatory insights were selected after three progressively deeper exploratory passes. The EDA notebooks are retained publicly in this repository so the analytical selection process can be audited without turning this report into an exploratory notebook.

- [Forensic EDA v1 — structure, quality and broad screening](https://github.com/stanleymay20/-M512-Hotel-Booking-Reliability/blob/main/eda/M512_Hotel_EDA_Forensic_v1.ipynb)
- [Forensic EDA v2 — source semantics, robustness and multivariable checks](https://github.com/stanleymay20/-M512-Hotel-Booking-Reliability/blob/main/eda/M512_Hotel_EDA_Forensic_v2.ipynb)
- [Forensic EDA v3 — duplicate sensitivity, subgroup stability and cancellation timing](https://github.com/stanleymay20/-M512-Hotel-Booking-Reliability/blob/main/eda/M512_Hotel_EDA_Forensic_v3_DEEP_DIVE.ipynb)
- [v3 findings memo](https://github.com/stanleymay20/-M512-Hotel-Booking-Reliability/blob/main/eda/M512_Hotel_EDA_Forensic_v3_Findings_Memo.md)
- [duplicate/timing forensic addendum](https://github.com/stanleymay20/-M512-Hotel-Booking-Reliability/blob/main/eda/M512_Hotel_EDA_v3_Forensic_Addendum_Duplicates_and_Timing.md)

All statistics shown below are recalculated in this notebook, so the assessed report remains self-contained.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates=[Path("hotel_bookings.csv"),Path("/mnt/data/hotelproj/hotel_bookings.csv")]
CSV=next((p for p in candidates if p.exists()),None)
if CSV is None:
    import kagglehub
    folder=Path(kagglehub.dataset_download("jessemostipak/hotel-booking-demand"))
    CSV=next(iter(folder.rglob("hotel_bookings.csv")))
df=pd.read_csv(CSV)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

## 1. Data quality and analytical boundaries

Missingness is concentrated in company (~94%) and agent (~14%); country has limited missingness and children has four missing values. There are 31,994 exact duplicate-looking rows, but no unique booking identifier, so identical anonymised rows cannot safely be declared erroneous duplicates. Source rows are retained; duplicate removal is used only as sensitivity analysis.

Outcome fields are not used as explanatory drivers. The data are observational: associations prioritise investigation, not causation.

In [ ]:
month_num={"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,"July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["arrival_month_num"]=df["arrival_date_month"].map(month_num)
df["arrival_date"]=pd.to_datetime(dict(year=df.arrival_date_year,month=df.arrival_month_num,day=df.arrival_date_day_of_month))
df["reservation_status_date"]=pd.to_datetime(df["reservation_status_date"])
df["stay_nights"]=df.stays_in_weekend_nights+df.stays_in_week_nights
df["guests"]=df.adults+df.children.fillna(0)+df.babies
df["has_prior_success"]=df.previous_bookings_not_canceled.gt(0)
bins=[-1,7,30,90,180,365,np.inf]
labels=["0–7","8–30","31–90","91–180","181–365","366+"]
df["lead_band"]=pd.cut(df.lead_time,bins,labels=labels,ordered=True)

def clean(ax):
    ax.spines[["top","right","left"]].set_visible(False)
    ax.grid(axis="y",alpha=.13)

## 2. Insight 1 — booking failure is material

**37.0% of supplied booking records were cancelled (44,224 of 119,390).**

In [ ]:
cancelled=int(df.is_canceled.sum()); total=len(df); rate=df.is_canceled.mean()*100
fig=plt.figure(figsize=(10,4.6)); ax=fig.add_axes([0,0,1,1]); ax.axis("off")
ax.text(.06,.84,"Booking failure is large enough to demand management attention",fontsize=19,weight="bold")
ax.text(.06,.70,"More than one in three supplied booking records were cancelled.",fontsize=12.5)
ax.text(.06,.40,f"{rate:.1f}%",fontsize=64,weight="bold",va="center")
ax.text(.06,.22,"cancelled",fontsize=20,weight="bold")
ax.text(.06,.08,f"{cancelled:,} of {total:,} records | Source: Hotel Booking Demand",fontsize=9)
plt.show()

## 3. Insight 2 — lead time is a strong reliability signal

Cancellation rises from **9.6% within seven days to 67.7% beyond one year**. The direction persisted across hotels and years in the forensic EDA.

**Implication:** test earlier confirmation or guarantee policies for long-lead bookings, without claiming causation.

In [ ]:
lead=df.groupby("lead_band",observed=True).is_canceled.agg(["size","mean"]).reset_index()
lead["rate"]=lead["mean"]*100
x=np.arange(len(lead))
fig,ax=plt.subplots(figsize=(10,5.4))
ax.plot(x,lead.rate,marker="o",linewidth=2.8)
ax.set_xticks(x); ax.set_xticklabels(lead.lead_band.astype(str)); ax.set_ylim(0,75)
ax.set_title("Booking reliability deteriorates sharply as lead time grows",loc="left",fontsize=17,weight="bold",pad=16)
for i in [0,2,5]:
    ax.annotate(f"{lead.loc[i,'rate']:.1f}%",(x[i],lead.loc[i,"rate"]),xytext=(0,8),textcoords="offset points",ha="center",weight="bold")
ax.set_xlabel("Days booked before arrival"); clean(ax); plt.tight_layout(); plt.show()

## 4. Insight 3 — cancellation exposure is concentrated

Online TA, Groups and Offline TA/TO generate **93.0% of all cancellations**.

In [ ]:
seg=df.groupby("market_segment").is_canceled.agg(bookings="size",cancellations="sum",cancel_rate="mean").reset_index()
seg["cancel_rate"]*=100
seg=seg[seg.bookings>=100].sort_values("cancellations")
top3=seg.nlargest(3,"cancellations")
names=top3.market_segment.tolist()
share=top3.cancellations.sum()/df.is_canceled.sum()*100
fig,ax=plt.subplots(figsize=(10,5.7))
bars=ax.barh(seg.market_segment,seg.cancellations)
for b,n in zip(bars,seg.market_segment): b.set_alpha(.9 if n in names else .22)
ax.set_title("Three market segments account for almost all cancellation volume",loc="left",fontsize=17,weight="bold")
for y,r in enumerate(seg.itertuples(index=False)):
    ax.text(r.cancellations+120,y,f"{int(r.cancellations):,} | {r.cancel_rate:.0f}% rate",va="center",fontsize=9)
ax.set_xlabel("Cancelled bookings"); ax.spines[["top","right","left"]].set_visible(False); plt.tight_layout(); plt.show()

## 5. Insight 4 — hotel context changes segment reliability

Groups and Offline TA/TO are less reliable at City Hotel in the source rows. Duplicate-removal sensitivity reduces the effect sizes, so this is contextual heterogeneity rather than a fixed causal effect.

In [ ]:
sh=df.groupby(["market_segment","hotel"]).is_canceled.agg(rate="mean",n="size").reset_index()
sh["rate"]*=100
p=sh.pivot(index="market_segment",columns="hotel",values="rate")
n=sh.pivot(index="market_segment",columns="hotel",values="n")
eligible=[s for s in p.index if pd.notna(p.loc[s].get("City Hotel",np.nan)) and pd.notna(p.loc[s].get("Resort Hotel",np.nan)) and n.loc[s,"City Hotel"]>=200 and n.loc[s,"Resort Hotel"]>=200]
focus={"Groups","Offline TA/TO"}
fig,ax=plt.subplots(figsize=(10,6.1))
for s in eligible:
    a,b=p.loc[s,"City Hotel"],p.loc[s,"Resort Hotel"]; strong=s in focus
    ax.plot([0,1],[a,b],marker="o",linewidth=3 if strong else 1.2,alpha=.95 if strong else .16)
    if strong:
        ax.text(-.03,a,f"{s}  {a:.0f}%",ha="right",va="center",weight="bold")
        ax.text(1.03,b,f"{b:.0f}%  {s}",ha="left",va="center",weight="bold")
ax.set_xlim(-.45,1.45); ax.set_ylim(0,75); ax.set_xticks([0,1]); ax.set_xticklabels(["City Hotel","Resort Hotel"],weight="bold")
ax.set_title("Hotel context changes the reliability of the same market segment",loc="left",fontsize=17,weight="bold")
ax.spines[["top","right","left","bottom"]].set_visible(False); ax.tick_params(axis="y",left=False,labelleft=False); plt.tight_layout(); plt.show()

## 6. Insight 5 — cancellation probability is not operational harm

Among records ending as Canceled, the median cancellation occurs **56 days before arrival**. Only **14.3% of cancelled/no-show failures are arrival-proximate**: cancellation within seven days or a no-show.

In [ ]:
cancel=df[df.reservation_status.eq("Canceled")].copy()
cancel["notice_days"]=(cancel.arrival_date-cancel.reservation_status_date).dt.days
early=int((cancel.notice_days>7).sum())
late=int(cancel.notice_days.between(0,7,inclusive="both").sum())
noshow=int(df.reservation_status.eq("No-Show").sum())
parts=pd.Series({"Cancelled >7 days":early,"Cancelled 0–7 days":late,"No-show":noshow})
shares=parts/parts.sum()*100
fig,ax=plt.subplots(figsize=(10,4.7)); left=0
for label,s in shares.items():
    ax.barh(["Booking failures"],[s],left=left,label=label)
    if s>5: ax.text(left+s/2,0,f"{s:.1f}%",ha="center",va="center",weight="bold")
    left+=s
ax.set_xlim(0,100); ax.set_title("Most booking failures happen before the final week",loc="left",fontsize=17,weight="bold")
ax.legend(frameon=False,loc="upper left",bbox_to_anchor=(0,-.2)); plt.tight_layout(); plt.show()

## 7. Insight 6 — Online TA dominates failure close to arrival

Late cancellations plus no-shows show a different priority: **Online TA contributes 47.1% of arrival-proximate failures**.

In [ ]:
cancel["late7"]=cancel.notice_days.between(0,7,inclusive="both")
late_by_seg=cancel.groupby("market_segment").late7.sum()
noshow_by_seg=df[df.reservation_status.eq("No-Show")].market_segment.value_counts()
prox=pd.DataFrame({"late":late_by_seg,"noshow":noshow_by_seg}).fillna(0)
prox["failures"]=prox.sum(axis=1); prox=prox[prox.failures>0]
prox["share"]=prox.failures/prox.failures.sum()*100; prox=prox.sort_values("failures")
fig,ax=plt.subplots(figsize=(10,5.4))
bars=ax.barh(prox.index,prox.failures); mx=prox.failures.idxmax()
for b,nm in zip(bars,prox.index): b.set_alpha(.9 if nm==mx else .25)
ax.set_title("Online TA drives nearly half of arrival-proximate booking failures",loc="left",fontsize=17,weight="bold")
for y,(nm,r) in enumerate(prox.iterrows()): ax.text(r.failures+25,y,f"{int(r.failures):,} | {r.share:.1f}%",va="center")
ax.set_xlabel("Arrival-proximate failures"); ax.spines[["top","right","left"]].set_visible(False); plt.tight_layout(); plt.show()

## 8. Insight 7 — recorded prior success marks more reliable demand

Bookings with recorded prior non-cancelled booking history cancel far less often. The source definition allows zero to include bookings without an associated customer profile, so this is a reliability marker rather than proof of a loyalty effect.

In [ ]:
hist=df.groupby("has_prior_success").is_canceled.agg(["size","mean"]).reset_index()
hist["label"]=hist.has_prior_success.map({False:"No prior success recorded",True:"Prior success recorded"})
hist["rate"]=hist["mean"]*100
fig,ax=plt.subplots(figsize=(8.5,4.6)); ax.barh(hist.label,hist.rate)
for y,r in enumerate(hist.itertuples(index=False)): ax.text(r.rate+.8,y,f"{r.rate:.1f}% cancelled | n={r.size:,}",va="center")
ax.set_xlabel("Cancellation rate (%)"); ax.spines[["top","right","left"]].set_visible(False); plt.tight_layout(); plt.show()

## 9. Recommendations, critical evaluation and conclusion

The evidence indicates two management problems. Reliability risk is concentrated in long-lead bookings and particular segments, while timing risk is concentrated differently near arrival. Management should test targeted interventions rather than impose blanket restrictions.

Limitations include two hotels, partial 2015/2017 coverage, duplicate ambiguity, and the absence of realised resale/revenue data.

**Conclusion.** Booking failure has two dimensions: **likelihood and timing**. Separating them gives management a more precise basis for targeted experimentation.

## References

Antonio, N., de Almeida, A. and Nunes, L. (2019) ‘Hotel booking demand datasets’, *Data in Brief*, 22, pp. 41–49.

Knaflic, C.N. (2015) *Storytelling with Data: A Data Visualization Guide for Business Professionals*. Hoboken, NJ: Wiley.

Mostipak, J. (n.d.) *Hotel Booking Demand*. Kaggle. Available at: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand (Accessed: 18 September 2026).